# Non-Reversible Anchored Langevin Dynamics (NALD)
## Elliptical Laplace and $\ell^1$ Laplace targets: $J=0$ vs. constant $J_a$ vs. state-dependent $J_s$

---

### The dynamics

Let $E=\mathbb{R}^d$ and let the target be

$$\pi(dx)\;\propto\;e^{-U(x)}\,dx ,$$

where $U$ is only **locally Lipschitz** — in particular $\nabla U$ need not exist everywhere.
Choose a $C^2$ **anchor** $U_0$ and a $C^2$ scalar field $\psi$, and set

$$a(x):=e^{(U-U_0)(x)},\qquad
  c(x):=e^{U(x)}J(x)\nabla\psi(x),\qquad
  b_\alpha(x):=-a(x)\nabla U_0(x)+\alpha\,c(x).$$

The **non-reversible anchored Langevin dynamics** is

$$dX_t=b_\alpha(X_t)\,dt+\sqrt{2a(X_t)}\,dW_t. \tag{NALD}$$

With the canonical choice $\psi=e^{-U_0}$ we get $c=-a\,J\nabla U_0$ and (NALD) becomes

$$dX_t=-a(X_t)\bigl(I_d+\alpha J(X_t)\bigr)\nabla U_0(X_t)\,dt+\sqrt{2a(X_t)}\,dW_t. \tag{NALD-c}$$

This is the form used throughout this notebook.

### Why $\pi\propto e^{-U}$ is invariant

The stationary Fokker–Planck equation for (NALD) with scalar diffusion $a$ is
$\nabla\cdot\bigl(-b_\alpha\rho+\nabla(a\rho)\bigr)=0$. Put $\rho=e^{-U}$. Then

* **Anchored (reversible) part.** $a\rho=e^{-U_0}$, hence $\nabla(a\rho)=-e^{-U_0}\nabla U_0$, while
  $(-a\nabla U_0)\rho=-e^{-U_0}\nabla U_0$. The two cancel **exactly**:
  $$-(-a\nabla U_0)\rho+\nabla(a\rho)=e^{-U_0}\nabla U_0-e^{-U_0}\nabla U_0=0 .$$
  The probability flux of the reversible part vanishes pointwise, *for any* $C^2$ anchor $U_0$.
  Only $\nabla U_0$ is ever evaluated — $\nabla U$ is never needed. This is the whole point of *anchoring*.

* **Antisymmetric part.** $\alpha c\,\rho=\alpha e^{U}J\nabla\psi\,e^{-U}=\alpha J\nabla\psi$, so the
  perturbation is invariance-preserving iff
  $$\nabla\cdot\bigl(J(x)\nabla\psi(x)\bigr)=0 .$$
  Writing it out, $\sum_{i,j}\partial_i(J_{ij}\partial_j\psi)=\sum_j\bigl(\sum_i\partial_iJ_{ij}\bigr)\partial_j\psi+\sum_{i,j}J_{ij}\partial_i\partial_j\psi$.
  The second sum vanishes because $J$ is skew and $\nabla^2\psi$ is symmetric. So the requirement on $J$ is
  $$J(x)^\top=-J(x)\quad\text{and}\quad \sum_i\partial_i J_{ij}(x)=0\ \ \forall j
  \qquad\text{(\emph{state-dependent-}}J\text{ assumptions).}$$
  For a **constant** skew $J$ the divergence condition is automatic.

Two consequences we will verify numerically and that drive the whole design:

1. **$U_0$ contributes no bias.** Any $C^2$ anchor gives the *same* invariant law $\pi\propto e^{-U}$;
   $U_0$ only affects efficiency. We exploit this by taking $U_0$ to be a **smoothing** of the
   non-differentiable $U$, which makes $a=e^{U-U_0}$ bounded above and below and $\nabla U_0$ Lipschitz.
2. **$\alpha$ and $J$ contribute no bias either** (at the level of the SDE): they only reshape the
   dynamics. Any deviation we observe in the samples is *discretisation* error, and we measure it separately.

### Plan

| Section | Content |
|---|---|
| 1 | Target **A**: elliptical Laplace — definitions of $U$ and $U_0$ |
| 2 | Target **B**: $\ell^1$ Laplace — definitions of $U$ and $U_0$ |
| 3 | The three perturbations $J=0$, $J_a$ (constant skew), $J_s(x)$ (state dependent) + verification |
| 4 | Integrator and diagnostics |
| 5 | Experiment 0 — correctness: all three $J$ hit the exact non-smooth target |
| 6 | Experiment 1 — discretisation bias vs. $\alpha$ and $h$ (and an instability of naive Euler for $J_s$) |
| 7 | Experiment 2 — mixing comparison at matched cost and matched accuracy |
| 8 | Experiment 3 — *why* $J_s$ behaves the way it does |
| 9 | Summary |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import time, math, warnings

%matplotlib inline
np.set_printoptions(precision=4, suppress=True, linewidth=140)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "figure.facecolor": "white",
})

D = 3                      # J_s below is a 3x3 field, so we work in d = 3 throughout
COL = {"J0": "#4C566A", "Ja": "#BF616A", "Js": "#5E81AC"}
print("numpy", np.__version__)

---
# 1. Target A — Elliptical Laplace

### Definition of $U$ and $U_0$

Fix a symmetric positive definite $\Sigma\in\mathbb{R}^{d\times d}$ and write
$q(x)=x^\top\Sigma^{-1}x$. The **elliptical (multivariate) Laplace** target is

$$\boxed{\;U(x)\;=\;\sqrt{x^\top\Sigma^{-1}x}\;=\;\sqrt{q(x)}\;},
\qquad \pi_A(x)\propto \exp\!\bigl(-\sqrt{x^\top\Sigma^{-1}x}\bigr).$$

$U$ is globally Lipschitz ($\|\nabla U\|_{\Sigma}\equiv 1$ a.e.) but **not differentiable at $x=0$**,
where the density has a conical cusp.

As anchor we take the *hyperbolic (pseudo-Huber) smoothing* of the same function, with $\delta>0$:

$$\boxed{\;U_0(x)\;=\;\sqrt{x^\top\Sigma^{-1}x+\delta^2}\;=\;\sqrt{q(x)+\delta^2}\;}\in C^\infty(\mathbb{R}^d),
\qquad
\nabla U_0(x)=\frac{\Sigma^{-1}x}{\sqrt{q(x)+\delta^2}} .$$

Consequently

$$a(x)=e^{U-U_0}=\exp\Bigl(-\tfrac{\delta^2}{\;\sqrt{q}+\sqrt{q+\delta^2}\;}\Bigr)\in\bigl[e^{-\delta},\,1\bigr],$$

so $a$ is **bounded away from $0$ and $\infty$** (uniform ellipticity), $\nabla U_0$ is bounded
by $\lambda_{\min}(\Sigma)^{-1/2}$ and globally Lipschitz, and hence $b_\alpha$ is locally Lipschitz
with sub-linear growth: Assumption *(Local regularity and well-posedness)* holds and the diffusion is
non-explosive.

**Exact reference.** With $y=\Sigma^{-1/2}x$ the density is $\propto e^{-\|y\|}$, so
$\|y\|\sim\mathrm{Gamma}(d,1)$ and $y/\|y\|$ is uniform on $S^{d-1}$, independent. Therefore
$$\mathbb{E}[xx^\top]=(d+1)\,\Sigma,\qquad \mathbb{E}[U(X)]=\mathbb{E}\|y\|=d ,$$
and we can draw i.i.d. samples from $\pi_A$ for ground truth.

In [ ]:
class EllipticLaplace:
    r"""Target A.   U(x)  = sqrt(x' S^{-1} x)                (elliptical Laplace, cusp at 0)
                      U0(x) = sqrt(x' S^{-1} x + delta^2)     (C^infty hyperbolic anchor)"""
    key, label = "A", "Elliptical Laplace"

    def __init__(self, Sigma, delta):
        self.Sigma = np.asarray(Sigma, float)
        self.d     = self.Sigma.shape[0]
        self.delta = float(delta)
        self.Sinv  = np.linalg.inv(self.Sigma)
        self.L     = np.linalg.cholesky(self.Sigma)          # L L' = Sigma

    # --- q(x) = x' Sigma^{-1} x -------------------------------------------------
    def q(self, x):      return np.einsum('...i,ij,...j->...', x, self.Sinv, x)

    # --- the two potentials -----------------------------------------------------
    def U(self, x):      return np.sqrt(self.q(x))                       #  NOT differentiable at 0
    def U0(self, x):     return np.sqrt(self.q(x) + self.delta**2)       #  C^infty anchor

    # --- everything the sampler needs ------------------------------------------
    def gradU0(self, x): return (x @ self.Sinv) / np.sqrt(self.q(x) + self.delta**2)[..., None]

    def log_a(self, x):                                  # log a = U - U0, stable form
        r = np.sqrt(self.q(x))
        return -self.delta**2 / (r + np.sqrt(r * r + self.delta**2))

    # --- exact i.i.d. sampling and exact moments (ground truth) -----------------
    def sample(self, n, rng):
        r = rng.gamma(self.d, 1.0, size=n)
        u = rng.standard_normal((n, self.d)); u /= np.linalg.norm(u, axis=1, keepdims=True)
        return (r[:, None] * u) @ self.L.T

    def cov_exact(self):  return (self.d + 1) * self.Sigma
    def EU_exact(self):   return float(self.d)
    def a_bounds(self):   return math.exp(-self.delta), 1.0


SIGMA_A = np.diag([9.0, 1.0, 0.25])          # anisotropic: Cov = 4*Sigma = diag(36, 4, 1)
DELTA_A = 0.1
tgtA = EllipticLaplace(SIGMA_A, DELTA_A)

print("Target A :", tgtA.label)
print("  Sigma          =", np.diag(SIGMA_A), "(diagonal)")
print("  delta          =", DELTA_A)
print("  exact Cov      =", np.diag(tgtA.cov_exact()), "  (marginal sd =", np.round(np.sqrt(np.diag(tgtA.cov_exact())),3), ")")
print("  exact E[U]     =", tgtA.EU_exact())
print("  a(x) range     = [%.4f, %.4f]" % tgtA.a_bounds())

In [ ]:
# Sanity: the exact sampler really is pi_A  (self-normalised importance check against e^{-U})
rng = np.random.default_rng(0)
xs  = tgtA.sample(400_000, rng)
print("empirical Cov vs exact  (max abs err):", np.abs(np.cov(xs.T) - tgtA.cov_exact()).max())
print("empirical E[U] = %.4f   exact = %.4f" % (tgtA.U(xs).mean(), tgtA.EU_exact()))
print("E||x||^2       = %.4f" % (xs**2).sum(1).mean())
print("a(x) observed range = [%.4f, %.4f]" % (np.exp(tgtA.log_a(xs)).min(), np.exp(tgtA.log_a(xs)).max()))

---
# 2. Target B — $\ell^1$ Laplace

### Definition of $U$ and $U_0$

With scale parameters $b=(b_1,\dots,b_d)$, $b_i>0$, the **$\ell^1$ (product) Laplace** target is

$$\boxed{\;U(x)\;=\;\sum_{i=1}^{d}\frac{|x_i|}{b_i}\;},
\qquad \pi_B(x)\propto\exp\Bigl(-\sum_i |x_i|/b_i\Bigr)=\prod_i \mathrm{Laplace}(x_i;0,b_i).$$

Here the non-differentiability is much more severe than in Target A: $\nabla U$ fails to exist on the
whole union of coordinate hyperplanes $\bigcup_i\{x_i=0\}$, a set that the chain approaches constantly
(it passes through it $O(1)$ times per unit time in every coordinate).

The anchor smooths each kink separately:

$$\boxed{\;U_0(x)\;=\;\sum_{i=1}^{d}\frac{\sqrt{x_i^2+\delta^2}}{b_i}\;}\in C^\infty(\mathbb{R}^d),
\qquad
\bigl(\nabla U_0(x)\bigr)_i=\frac{x_i}{b_i\sqrt{x_i^2+\delta^2}} .$$

Then
$$a(x)=\exp\Bigl(-\sum_i\frac{\delta^2}{b_i\bigl(|x_i|+\sqrt{x_i^2+\delta^2}\bigr)}\Bigr)
      \in\Bigl[\exp\bigl(-\delta\textstyle\sum_i b_i^{-1}\bigr),\,1\Bigr],$$
$\|\nabla U_0\|_\infty\le\max_i b_i^{-1}$, and $\nabla U_0$ is Lipschitz with constant
$\max_i (b_i\delta)^{-1}$.

**Exact reference.** $x_i\sim\mathrm{Laplace}(0,b_i)$ independently, so
$\mathrm{Var}(x_i)=2b_i^2$, $\mathbb{E}|x_i|=b_i$, hence $\mathbb{E}[U(X)]=d$, and i.i.d. sampling is trivial.

In [ ]:
class L1Laplace:
    r"""Target B.   U(x)  = sum_i |x_i| / b_i                       (kinks on all coordinate planes)
                      U0(x) = sum_i sqrt(x_i^2 + delta^2) / b_i      (C^infty anchor)"""
    key, label = "B", r"$\ell^1$ Laplace"

    def __init__(self, b, delta):
        self.b     = np.asarray(b, float)
        self.d     = len(self.b)
        self.delta = float(delta)

    # --- the two potentials -----------------------------------------------------
    def U(self, x):      return np.sum(np.abs(x) / self.b, axis=-1)                     # non-smooth
    def U0(self, x):     return np.sum(np.sqrt(x * x + self.delta**2) / self.b, axis=-1)  # C^infty

    # --- everything the sampler needs ------------------------------------------
    def gradU0(self, x): return x / (self.b * np.sqrt(x * x + self.delta**2))

    def log_a(self, x):                                  # log a = U - U0, stable form
        ax = np.abs(x)
        return -np.sum(self.delta**2 / (self.b * (ax + np.sqrt(ax * ax + self.delta**2))), axis=-1)

    # --- exact i.i.d. sampling and exact moments (ground truth) -----------------
    def sample(self, n, rng): return rng.laplace(0.0, self.b, size=(n, self.d))
    def cov_exact(self):      return np.diag(2 * self.b**2)
    def EU_exact(self):       return float(self.d)
    def a_bounds(self):       return math.exp(-self.delta * np.sum(1 / self.b)), 1.0


B_B     = np.array([2.0, 1.0, 0.4])          # anisotropic scales
DELTA_B = 0.1
tgtB = L1Laplace(B_B, DELTA_B)

print("Target B :", "l1 Laplace")
print("  b              =", B_B)
print("  delta          =", DELTA_B)
print("  exact Var      =", np.diag(tgtB.cov_exact()), "  (marginal sd =", np.round(np.sqrt(np.diag(tgtB.cov_exact())),3), ")")
print("  exact E[U]     =", tgtB.EU_exact())
print("  a(x) range     = [%.4f, %.4f]" % tgtB.a_bounds())

xs = tgtB.sample(400_000, np.random.default_rng(1))
print("  empirical Var  =", np.round(np.var(xs, 0), 4), " E[U] = %.4f" % tgtB.U(xs).mean())
print("  E||x||^2       = %.4f" % (xs**2).sum(1).mean())

TARGETS = {"A": tgtA, "B": tgtB}

In [ ]:
# A picture of the two targets: exact draws + the level sets of U (non-smooth) and U0 (smooth)
fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.4))
grid = np.linspace(-9, 9, 401)
for row, (tg, ttl, lim) in enumerate([(tgtA, "A: elliptical Laplace", 9), (tgtB, r"B: $\ell^1$ Laplace", 7)]):
    X1, X2 = np.meshgrid(np.linspace(-lim, lim, 401), np.linspace(-lim, lim, 401))
    P = np.stack([X1, X2, np.zeros_like(X1)], -1)
    axes[row,0].contour(X1, X2, tg.U(P), levels=12, colors="#BF616A", linewidths=.8)
    axes[row,0].set_title(f"{ttl}\n$U(x)$ level sets (slice $x_3=0$)")
    axes[row,1].contour(X1, X2, tg.U0(P), levels=12, colors="#5E81AC", linewidths=.8)
    axes[row,1].set_title(f"anchor $U_0(x)$, $\\delta={tg.delta}$\n(smoothed level sets)")
    s = tg.sample(20000, np.random.default_rng(2))
    axes[row,2].plot(s[:,0], s[:,1], '.', ms=.6, alpha=.25, color="#2E3440")
    axes[row,2].set_title("exact i.i.d. draws from $\\pi$\n(projection on $(x_1,x_2)$)")
    for a_ in axes[row]:
        a_.set_xlim(-lim, lim); a_.set_ylim(-lim, lim); a_.set_aspect('equal')
        a_.set_xlabel("$x_1$"); a_.set_ylabel("$x_2$")
fig.tight_layout(); plt.show()

In [ ]:
# The anchor is a *smoothing*: U - U0 is tiny and bounded, which keeps a = e^{U-U0} = O(1).
fig, axes = plt.subplots(1, 2, figsize=(9, 3.0))
t = np.linspace(-3, 3, 1201)
for ax, tg, ttl in [(axes[0], tgtA, "Target A"), (axes[1], tgtB, r"Target B")]:
    P = np.stack([t, np.zeros_like(t), np.zeros_like(t)], -1)
    ax.plot(t, tg.U(P),  color="#BF616A", lw=1.4, label="$U$  (non-smooth)")
    ax.plot(t, tg.U0(P), color="#5E81AC", lw=1.4, ls="--", label="$U_0$  (anchor, $C^2$)")
    ax.plot(t, np.exp(tg.log_a(P)), color="#A3BE8C", lw=1.4, label="$a=e^{U-U_0}$")
    lo, hi = tg.a_bounds(); ax.axhline(lo, color="#A3BE8C", lw=.6, ls=":")
    ax.set_title(f"{ttl} along $x_1$ (others $=0$);  $a\\in[{lo:.3f},1]$")
    ax.set_xlabel("$x_1$"); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

---
# 3. The three perturbations

All three choices are used inside the canonical form (NALD-c),
$b_\alpha=-a\,(I+\alpha J)\nabla U_0$, so that $\alpha$ is the single knob controlling the strength of
the non-reversible part.

**(i) $J=0$ — reversible anchored Langevin.** The baseline,
$dX_t=-a\nabla U_0\,dt+\sqrt{2a}\,dW_t$.

**(ii) $J_a$ — constant skew-symmetric.** In $d=3$ every skew matrix is a cross-product operator,
$J_a v=u\times v$ for some $u\in\mathbb{R}^3$. We take the "democratic" axis
$u=(1,1,1)/\sqrt3$:

$$J_a=\widehat{u}=\begin{pmatrix}0&-u_3&u_2\\ u_3&0&-u_1\\ -u_2&u_1&0\end{pmatrix},
\qquad u=\tfrac{1}{\sqrt3}(1,1,1)^\top .$$

Being constant it is trivially divergence free, $\sum_i\partial_i (J_a)_{ij}=0$.

**(iii) $J_s(x)$ — state dependent.**

$$J_s(x)=\begin{pmatrix}0&-sx_3&sx_2\\ sx_3&0&-sx_1\\ -sx_2&sx_1&0\end{pmatrix}=s\,\widehat{x},
\qquad\text{i.e.}\qquad J_s(x)v=s\,(x\times v).$$

It is skew by inspection, and column by column
$$\sum_i\partial_i (J_s)_{i1}=\partial_2(sx_3)+\partial_3(-sx_2)=0,\quad
  \sum_i\partial_i (J_s)_{i2}=\partial_1(-sx_3)+\partial_3(sx_1)=0,\quad
  \sum_i\partial_i (J_s)_{i3}=\partial_1(sx_2)+\partial_2(-sx_1)=0,$$
so the state-dependent-$J$ assumptions hold and $\pi\propto e^{-U}$ stays invariant.

**Scaling convention.** $J_a$ satisfies $\|J_a v\|\le\|v\|$, whereas $\|J_s(x)v\|\le s\|x\|\|v\|$ grows
with $\|x\|$. To make $\alpha$ mean the same thing for both we fix
$$s:=\bigl(\mathbb{E}_\pi\|X\|^2\bigr)^{-1/2},$$
so that $\|J_s(X)\|\approx1$ for a typical draw. Only the product $\alpha s$ matters for $J_s$, so
sweeping $\alpha$ sweeps the whole family.

**Geometric remark (used in §6 and §8).** With the canonical $\psi=e^{-U_0}$,
$$\alpha c(x)=-\alpha\,a\,J_s\nabla U_0=-\alpha s\,a\,\bigl(x\times\nabla U_0\bigr)
             =\underbrace{\bigl(\alpha s\,a\,\nabla U_0\bigr)}_{=:\;\omega(x)}\times\,x .$$
So the $J_s$ drift is an **instantaneous rigid rotation of $x$ about the axis $\nabla U_0(x)$**, and its
flow conserves $\|x\|$ exactly. Two things follow: the flow can be integrated exactly by Rodrigues'
formula (which we will need, because the naive Euler step does *not* conserve $\|x\|$ and blows up),
and the perturbation **vanishes wherever $x\parallel\nabla U_0(x)$**.

In [ ]:
def hat(u):
    """Cross-product (skew) matrix:  hat(u) v = u x v."""
    u = np.asarray(u, float)
    return np.array([[0.0, -u[2],  u[1]],
                     [u[2],  0.0, -u[0]],
                     [-u[1], u[0],  0.0]])

U_AXIS = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)
J_A    = hat(U_AXIS)                      # constant skew-symmetric perturbation

# s such that ||J_s(X)|| ~ 1 for a typical draw from each target
S_STATE = {}
for k, tg in TARGETS.items():
    S_STATE[k] = float(1.0 / np.sqrt((tg.sample(400_000, np.random.default_rng(5))**2).sum(1).mean()))

print("J_a =\n", J_A)
print("\ns (state-dependent scale)  A: %.4f   B: %.4f" % (S_STATE["A"], S_STATE["B"]))

In [ ]:
# ---- verification that both J's satisfy the assumptions -------------------------------------
def J_s_matrix(x, s):
    """The 3x3 matrix exactly as written in the text."""
    return np.array([[0.0,   -s*x[2],  s*x[1]],
                     [s*x[2],  0.0,   -s*x[0]],
                     [-s*x[1], s*x[0],  0.0  ]])

rng = np.random.default_rng(7)
s   = S_STATE["A"]

# (1) skew-symmetry
err_skew_a = np.abs(J_A + J_A.T).max()
err_skew_s = max(np.abs(J_s_matrix(x, s) + J_s_matrix(x, s).T).max() for x in rng.standard_normal((200, 3)))

# (2) J_s(x) v == s * (x cross v)
err_cross = max(np.abs(J_s_matrix(x, s) @ v - s*np.cross(x, v)).max()
                for x, v in zip(rng.standard_normal((200,3)), rng.standard_normal((200,3))))

# (3) column-wise divergence  sum_i d_i J_ij = 0   (central differences)
def col_div(Jfun, x, eps=1e-5):
    out = np.zeros(3)
    for j in range(3):
        for i in range(3):
            xp, xm = x.copy(), x.copy(); xp[i] += eps; xm[i] -= eps
            out[j] += (Jfun(xp)[i, j] - Jfun(xm)[i, j]) / (2*eps)
    return out
err_div = max(np.abs(col_div(lambda z: J_s_matrix(z, s), x)).max() for x in rng.standard_normal((50,3)))

# (4) the invariance condition itself:  div( J(x) grad psi(x) ) = 0   with psi = exp(-U0)
def div_Jgradpsi(tg, Jfun, x, eps=1e-4):
    def field(z):
        gpsi = -tg.gradU0(z[None, :])[0] * np.exp(-tg.U0(z[None, :])[0])   # grad psi, psi = e^{-U0}
        return Jfun(z) @ gpsi
    tot = 0.0
    for i in range(3):
        xp, xm = x.copy(), x.copy(); xp[i] += eps; xm[i] -= eps
        tot += (field(xp)[i] - field(xm)[i]) / (2*eps)
    return tot

pts = rng.standard_normal((40, 3)) * 2.0
for k, tg in TARGETS.items():
    sk = S_STATE[k]
    dv_a = max(abs(div_Jgradpsi(tg, lambda z: J_A,               x)) for x in pts)
    dv_s = max(abs(div_Jgradpsi(tg, lambda z: J_s_matrix(z, sk), x)) for x in pts)
    scale = max(np.abs(np.exp(-tg.U0(pts))).max(), 1e-12)
    print(f"target {k}:  max |div(J grad psi)| ->  J_a: {dv_a:.2e}   J_s: {dv_s:.2e}   (psi scale ~ {scale:.2e})")

print("\nmax |J + J'|            J_a: %.2e   J_s: %.2e" % (err_skew_a, err_skew_s))
print("max |J_s(x)v - s x X v|            %.2e" % err_cross)
print("max |sum_i d_i J_s,ij|             %.2e   (finite-difference floor)" % err_div)

---
# 4. Integrator and diagnostics

### 4.1 Discretisation

The natural discretisation of (NALD-c) is Euler–Maruyama,

$$X_{n+1}=X_n-h\,a(X_n)\bigl(I+\alpha J(X_n)\bigr)\nabla U_0(X_n)+\sqrt{2\,a(X_n)h}\;\xi_n,
\qquad \xi_n\sim N(0,I_d).$$

For $J=0$ and $J=J_a$ we use exactly this. For $J=J_s$ we use it too (`rotation="euler"`), but it turns
out to be **unstable**: the exact $J_s$-flow $\dot x=\omega\times x$ conserves $\|x\|$, while one explicit
Euler step multiplies $\|x\|$ by $\sqrt{1+(h\|\omega\|)^2}$. The resulting systematic outward push grows
linearly in $\|x\|$ whereas the restoring drift $-a\nabla U_0$ is *bounded*, so beyond a radius
$r^\ast\approx 2/(h\|\omega\|^2)$ the chain escapes to infinity. We therefore also provide a
**Lie–Trotter splitting** (`rotation="exact"`, the default) in which the rotation is applied exactly via
Rodrigues' formula,

$$X_{n+1}=R\bigl(\omega_n,h\bigr)\,\Bigl[X_n-h\,a_n\nabla U_0(X_n)+\sqrt{2a_nh}\,\xi_n\Bigr],
\qquad \omega_n=\alpha\,s\,a_n\nabla U_0(X_n),$$

$$R(\omega,h)y=y\cos\theta+(k\times y)\sin\theta+k\,(k\!\cdot\!y)(1-\cos\theta),\qquad
k=\omega/\|\omega\|,\ \theta=h\|\omega\|.$$

Both schemes are weakly first-order in $h$, use **exactly one $\nabla U_0$ evaluation per step**, and
coincide with plain Euler–Maruyama when $J=0$ or $J=J_a$. The cost per step is therefore identical
across all three perturbations, and comparing integrated autocorrelation times *in steps* (equivalently
in units of simulated time, since $h$ is shared inside each experiment) is a fair comparison.

Neither scheme is Metropolis-corrected. A Metropolis accept/reject step would destroy the
non-reversibility that is the whole point, so instead we **measure** the discretisation bias directly
(§6) and only compare mixing in regimes where that bias is verified small.

### 4.2 Diagnostics

* **Integrated autocorrelation time** $\tau_f$ for a test function $f$, from the FFT estimate of the
  autocorrelation averaged over chains, truncated by Sokal's automatic window ($W\ge 5\tau$).
* **Effective sample size** $\mathrm{ESS}=K\,N/\tau$ for $K$ chains of $N$ samples.
* **Speed-up** $= \tau_f(J=0)\,/\,\tau_f(J)$ at equal cost.
* **Stationarity drift** (§6): start $M$ chains from *exact* i.i.d. draws $X_0\sim\pi$, run to time $T$,
  and report the paired relative change $\bigl(\mathbb{E}f(X_T)-\mathbb{E}f(X_0)\bigr)/\mathbb{E}f(X_0)$.
  If the scheme preserved $\pi$ this would be $0$; pairing makes it a very low-variance bias estimate.

In [ ]:
def _cross(a, b):
    out = np.empty_like(a)
    a0, a1, a2 = a[:,0], a[:,1], a[:,2]; b0, b1, b2 = b[:,0], b[:,1], b[:,2]
    out[:,0] = a1*b2 - a2*b1; out[:,1] = a2*b0 - a0*b2; out[:,2] = a0*b1 - a1*b0
    return out

def rodrigues(X, w, t):
    """Exact flow of  dX/dt = w x X  over time t (w frozen).  Norm preserving."""
    nrm  = np.sqrt(w[:,0]**2 + w[:,1]**2 + w[:,2]**2)
    th   = nrm * t
    k    = w / np.where(nrm > 1e-300, nrm, 1.0)[:, None]
    ct   = np.cos(th)[:, None]; st = np.sin(th)[:, None]
    return X*ct + _cross(k, X)*st + k*(k*X).sum(1)[:, None]*(1.0 - ct)


def nald(tgt, J="none", alpha=0.0, s=0.0, Ja=J_A, h=0.01, n_steps=100_000, n_chains=48,
         seed=0, burn=None, thin=10, x0=None, rotation="exact", block=2000,
         record=None, store=True):
    r"""Simulate  dX = -a(X)(I + alpha J(X)) grad U0(X) dt + sqrt(2 a(X)) dW   (canonical psi = e^{-U0}).

    J        : "none" | "const" (uses Ja) | "state" (uses s, J_s(x) = s*hat(x))
    rotation : for J="state" only -- "exact" (Rodrigues splitting) or "euler" (plain Euler-Maruyama)
    record   : callable  states (n_chains,d) -> (n_chains, m)  of test functions to store.
               Default records (x_1, ..., x_d, U(x)).
    Returns  (trace, X_final);  trace has shape (n_kept, n_chains, m).
    """
    rng = np.random.default_rng(seed)
    d   = tgt.d
    X   = tgt.sample(n_chains, rng) if x0 is None else np.array(x0, float, copy=True)
    n_chains = X.shape[0]
    if burn is None: burn = n_steps // 5
    if record is None:
        record = lambda z: np.concatenate([z, tgt.U(z)[:, None]], axis=1)
    sh = np.sqrt(2.0 * h)
    M  = np.ascontiguousarray(np.asarray(Ja, float).T)          # so that  g @ M == (Ja g)
    m  = record(X).shape[1]
    keep = ((n_steps - burn) + thin - 1)//thin if store else 0
    out  = np.empty((keep, n_chains, m)) if store else None
    kk, bi, nz = 0, block, None

    for n in range(n_steps):
        if bi == block:
            nz = rng.standard_normal((block, n_chains, d)); bi = 0
        a = np.exp(tgt.log_a(X))[:, None]
        g = tgt.gradU0(X)
        Y = X - h*(a*g) + (sh*np.sqrt(a))*nz[bi]; bi += 1
        if J == "const":
            Y -= (h*alpha) * (a * (g @ M))
        elif J == "state":
            w = (alpha*s) * (a*g)                                # dX/dt = w x X
            Y = rodrigues(Y, w, h) if rotation == "exact" else Y + h*_cross(w, Y)
        X = Y
        if store and n >= burn and (n - burn) % thin == 0:
            out[kk] = record(X); kk += 1
    return (out[:kk] if store else None), X

In [ ]:
# ---------------- autocorrelation / ESS ----------------
def _acf(y):
    n = len(y); m = 1
    while m < 2*n: m *= 2
    f  = np.fft.rfft(y - y.mean(), n=m)
    ac = np.fft.irfft(f*np.conj(f), n=m)[:n].real
    return ac / ac[0]

def acf_mean(trace_col):
    """trace_col: (n_samples, n_chains) -> chain-averaged autocorrelation function."""
    return np.mean([_acf(trace_col[:, k]) for k in range(trace_col.shape[1])], axis=0)

def iact_from_acf(ac, c=5.0):
    taus = 2*np.cumsum(ac) - 1.0
    for w in range(len(taus)):
        if w >= c*taus[w]:
            return max(float(taus[w]), 1.0)
    return max(float(taus[-1]), 1.0)

def summarize(trace, thin, h, names, exact=None):
    """trace (n,K,m) of recorded test functions -> dict of IACT (in steps and in time), ESS, moments."""
    n, K, m = trace.shape
    acfs = [acf_mean(trace[:, :, j]) for j in range(m)]
    tau_s = np.array([iact_from_acf(ac)*thin for ac in acfs])      # IACT in integrator steps
    res = dict(names=names, acf=acfs, thin=thin, h=h,
               tau_steps=tau_s, tau_time=tau_s*h,
               ess=np.array([n*K/(iact_from_acf(ac)) for ac in acfs]),
               mean=trace.reshape(-1, m).mean(0), var=trace.reshape(-1, m).var(0),
               n=n, K=K)
    if exact is not None: res["exact"] = exact
    return res

---
# 5. Experiment 0 — correctness

The claim to check first is the one that makes the method interesting: **NALD samples the exact
non-smooth target $\pi\propto e^{-U}$, even though only the smoothed $\nabla U_0$ is ever evaluated, and
for every admissible $J$.** We run each of the three perturbations at a small step size and compare
against exact i.i.d. draws.

In [ ]:
CFG0 = [("J0", dict(J="none"),                              r"$J=0$"),
        ("Ja", dict(J="const", alpha=2.0),                  r"$J_a$, $\alpha=2$"),
        ("Js", dict(J="state", alpha=2.0),                  r"$J_s$, $\alpha=2$")]

H0, NS0, NC0, THIN0 = 0.002, 600_000, 64, 20
exp0 = {}
for k, tg in TARGETS.items():
    exp0[k] = {}
    for tag, kw, lab in CFG0:
        t0 = time.time()
        kw = dict(kw); kw.setdefault("s", S_STATE[k])
        tr, _ = nald(tg, h=H0, n_steps=NS0, n_chains=NC0, seed=101, burn=NS0//6, thin=THIN0, **kw)
        exp0[k][tag] = dict(trace=tr, label=lab)
        print(f"target {k}  {tag:<3} {lab:<18} kept {tr.shape}  {time.time()-t0:5.1f}s")
print("\nsimulated time per chain: T = %.0f" % (NS0*H0))

In [ ]:
# Moment table with Monte-Carlo error bars from the effective sample size
LAB = {"J0": "J = 0", "Ja": "J_a  alpha=2", "Js": "J_s  alpha=2"}
names0 = ["x1", "x2", "x3", "U"]
for k, tg in TARGETS.items():
    ex_var = np.diag(tg.cov_exact()); ex = np.r_[np.zeros(3), tg.EU_exact()]
    print(f"\n=== target {k}: {'elliptical Laplace' if k=='A' else 'l1 Laplace'} "
          f"(h={H0}, T={NS0*H0:.0f}, {NC0} chains; +-2 s.e.) ===")
    print(f"{'':<14}" + "".join(f"{'E['+n+']':>19}" for n in names0)
                      + "".join(f"{'Var['+n+']':>19}" for n in names0[:3]))
    print(f"{'exact':<14}" + "".join(f"{v:>19.4f}" for v in ex)
                           + "".join(f"{v:>19.4f}" for v in ex_var))
    for tag, kw, lab in CFG0:
        tr = exp0[k][tag]["trace"]; S = summarize(tr, THIN0, H0, names0); f = tr.reshape(-1, 4)
        se = f.std(0) / np.sqrt(S["ess"])
        row_m = "".join(f"{f[:,j].mean():>11.4f} +-{2*se[j]:<6.3f}" for j in range(4))
        row_v = "".join(f"{f[:,j].var():>11.3f} +-{2*f[:,j].var()*np.sqrt(2/S['ess'][j]):<6.2f}" for j in range(3))
        print(f"{LAB[tag]:<14}" + row_m + row_v)

In [ ]:
# Marginal densities: NALD (three perturbations) vs exact i.i.d. draws
rng = np.random.default_rng(31)
for k, tg in TARGETS.items():
    exact = tg.sample(600_000, rng)
    fig, axes = plt.subplots(1, 4, figsize=(13, 2.7))
    for j in range(4):
        vals_ex = exact[:, j] if j < 3 else tg.U(exact)
        lo, hi  = np.quantile(vals_ex, [0.0005, 0.9995])
        bins    = np.linspace(lo, hi, 120)
        axes[j].hist(vals_ex, bins=bins, density=True, color="0.85", label="exact i.i.d.")
        for tag, kw, lab in CFG0:
            v = exp0[k][tag]["trace"][:, :, j].ravel()
            hh, _ = np.histogram(v, bins=bins, density=True)
            axes[j].plot(0.5*(bins[1:]+bins[:-1]), hh, lw=1.1, color=COL[tag], label=lab)
        axes[j].set_title(("$x_%d$" % (j+1)) if j < 3 else "$U(x)$")
        axes[j].set_yscale("log"); axes[j].set_ylim(bottom=max(1e-5, axes[j].get_ylim()[0]))
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Target {k} — marginals, log scale  (h={H0})", y=1.04)
    fig.tight_layout(); plt.show()

In [ ]:
# Q-Q plots against exact draws: the cusp region and the tails
probs = np.concatenate([np.linspace(1e-4, .02, 40), np.linspace(.02, .98, 120), np.linspace(.98, 1-1e-4, 40)])
for k, tg in TARGETS.items():
    exact = tg.sample(800_000, np.random.default_rng(32))
    fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
    for j in range(4):
        qe = np.quantile(exact[:, j] if j < 3 else tg.U(exact), probs)
        for tag, kw, lab in CFG0:
            qn = np.quantile(exp0[k][tag]["trace"][:, :, j].ravel(), probs)
            axes[j].plot(qe, qn, lw=1.0, color=COL[tag], label=lab)
        axes[j].plot(qe, qe, "k--", lw=.7)
        axes[j].set_title(("$x_%d$" % (j+1)) if j < 3 else "$U(x)$")
        axes[j].set_xlabel("exact quantile"); axes[j].set_ylabel("NALD quantile")
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Target {k} — Q-Q vs exact", y=1.05); fig.tight_layout(); plt.show()


### The anchor introduces no bias: varying $\delta$

$U_0$ is an arbitrary $C^2$ anchor, so the invariant law must not depend on $\delta$ — only the
efficiency does. We check this with the **paired stationarity drift**: start $M$ chains at exact draws
$X_0\sim\pi$, integrate to time $T$, and measure the relative change of $\mathbb{E}f$. For an
exactly $\pi$-preserving scheme this is $0$; what is left is the $O(h)$ discretisation error alone.
Every $\delta$ is run from the same seed, hence with the same initial draws and the same driving noise,
so the rows can be compared with each other far below their individual Monte-Carlo error.

The second table shows what $\delta$ *does* control. It is a genuine trade-off:

* small $\delta$ keeps $a=e^{U-U_0}$ near $1$ everywhere, so the diffusion is fast — but $\nabla U_0$
  has Lipschitz constant $O(1/\delta)$ near the kinks, so the integrator needs a smaller $h$;
* large $\delta$ gives a gentle, well-conditioned drift — but $a$ collapses (to $\sim0.02$ at
  $\delta=1$ for target B), throttling the diffusion and inflating $\tau$ several-fold.

We keep $\delta=0.1$ for both targets and pay for it with a smaller $h$ on target B.


In [ ]:
def stat_drift(tgt, T=2.0, h=0.005, M=40_000, seed=3, **kw):
    """Paired relative drift of E[f] after time T, starting from exact draws.  0 = exactly invariant."""
    rng = np.random.default_rng(seed)
    X0  = tgt.sample(M, rng)
    K   = int(round(T/h))
    _, XT = nald(tgt, h=h, n_steps=K, n_chains=M, seed=seed+1, x0=X0, store=False, **kw)
    F = lambda z: np.column_stack([z**2, tgt.U(z)[:, None]])
    F0, FT = F(X0), F(XT)
    Dd = FT - F0
    base = np.abs(F0.mean(0))
    return Dd.mean(0)/base, (Dd.std(0)/np.sqrt(M))/base


# (i) the invariant law does not depend on delta.  All rows use the SAME seed, hence the same
#     initial draws and the same driving noise, so the rows are directly comparable to each other
#     far below their individual Monte-Carlo error.
print("(i) stationarity drift of E[f] over T=2, J = 0, common random numbers across delta\n")
for k, cls, par, hh in [("A", EllipticLaplace, SIGMA_A, 0.005), ("B", L1Laplace, B_B, 0.0025)]:
    print(f"  target {k} (h={hh}):   {'delta':>7} {'a range':>17}   " + "".join(f"{n:>11}" for n in ["x1^2","x2^2","x3^2","U"]))
    for dl in [0.02, 0.1, 0.3, 1.0]:
        tg = cls(par, dl)
        m, se = stat_drift(tg, h=hh, M=120_000, J="none")
        lo, hi = tg.a_bounds()
        print(f"  {'':>18}{dl:>7} {'[%.3f, %.2f]'%(lo,hi):>17}   " + "".join(f"{v:>11.4f}" for v in m)
              + f"    (2se ~ {2*se.max():.4f})")
    print()

In [ ]:
# (ii) ... but delta is not free: it trades mixing speed against discretisation error.
#      a = e^{U-U0} shrinks as delta grows, which slows the diffusion everywhere,
#      while grad U0 becomes less stiff, which reduces the per-step error.
print("(ii) effect of delta on efficiency (J = 0)\n")
for k, cls, par, hh, ns in [("A", EllipticLaplace, SIGMA_A, 0.005, 400_000),
                            ("B", L1Laplace,       B_B,     0.0025, 500_000)]:
    print(f"  target {k} (h={hh}, {ns:,} steps, 48 chains)")
    print(f"    {'delta':>7} {'min a':>9} " + "".join(f"{'tau_'+n:>10}" for n in ["x1","x2","x3","U"])
          + f"{'E[U]':>10}{'max|drift|':>12}")
    for dl in [0.1, 0.3, 1.0]:
        tg = cls(par, dl)
        tr, _ = nald(tg, h=hh, n_steps=ns, n_chains=48, seed=5, burn=ns//5, thin=10)
        S  = summarize(tr, 10, hh, ["x1","x2","x3","U"])
        dm = np.abs(stat_drift(tg, h=hh, M=60_000, J="none")[0]).max()
        print(f"    {dl:>7} {tg.a_bounds()[0]:>9.4f} " + "".join(f"{v:>10.0f}" for v in S["tau_steps"])
              + f"{tr.reshape(-1,4)[:,3].mean():>10.4f}{dm:>12.4f}")
    print(f"    exact E[U] = {tg.EU_exact():.4f};  tau in integrator steps\n")
print("Small delta mixes faster (a stays near 1) but needs a smaller h; large delta is gentler on")
print("the integrator but throttles the diffusion.  We use delta = 0.1 for both targets and pay")
print("for it with h = 0.0025 on target B, whose kinks are far denser than target A's single cusp.")

---
# 6. Experiment 1 — discretisation bias, and an instability of naive Euler for $J_s$

Neither $J$ nor $\alpha$ biases the *continuous* dynamics, so everything below is discretisation error.
Two effects show up.

**(a) Bias grows with $\alpha$.** The perturbation adds a drift of size $\alpha\|c\|$; the Euler/splitting
local error is $O(h)$ with a constant that grows with the drift, so the usable $\alpha$ at fixed $h$ is
bounded. This is the fundamental trade-off of non-reversible samplers: stirring harder costs accuracy
unless $h$ shrinks proportionally.

**(b) Plain Euler–Maruyama is *unstable* for $J_s$.** As noted in §4.1, the exact $J_s$-flow conserves
$\|x\|$ but explicit Euler inflates it by $\sqrt{1+(h\|\omega\|)^2}$ per step, while the restoring drift
$-a\nabla U_0$ is bounded. Beyond $r^\ast\approx2/(h\|\omega\|^2)$ the chain runs away. The Rodrigues
splitting removes the instability entirely.

Because the diagnostic is itself a Monte-Carlo estimate, it has a noise floor of roughly
$2\,\mathrm{s.e.}\approx1.5\%$ at the sample sizes used below. We therefore flag a configuration as
untrustworthy only when its measured drift exceeds `TOL = 4%`, about three times that floor, and we
print the standard error next to every row so the floor is visible.


In [ ]:
# ---- (b) the instability, shown directly --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3.0))
np.seterr(over="ignore", invalid="ignore")
for ax, (k, tg) in zip(axes, TARGETS.items()):
    for rot, ls in [("euler", "--"), ("exact", "-")]:
        for alpha, cc in [(4.0, "#D08770"), (8.0, "#BF616A"), (16.0, "#8B2D3A")]:
            tr, _ = nald(tg, J="state", alpha=alpha, s=S_STATE[k], h=0.01, n_steps=120_000,
                         n_chains=64, seed=17, burn=0, thin=200,
                         record=lambda z: np.sqrt((z**2).sum(1))[:, None], rotation=rot)
            r = np.nanmax(np.abs(np.nan_to_num(tr[:, :, 0], nan=0.0, posinf=1e30)), axis=1)
            ax.semilogy(np.arange(len(r))*200*0.01, np.clip(r, 1e-3, 1e30), ls, color=cc, lw=1.0,
                        label=f"{'Euler' if rot=='euler' else 'Rodrigues'}, $\\alpha$={alpha:g}")
    ax.set_title(f"Target {k}: $\\max_k\\|X_t^{{(k)}}\\|$, $J_s$, h=0.01, 64 chains")
    ax.set_xlabel("simulated time $t$"); ax.set_ylabel(r"$\max_k\|X_t\|$")
    ax.legend(fontsize=6.5, ncol=2)
fig.tight_layout(); plt.show()
print("dashed = plain Euler-Maruyama (escapes to infinity);  solid = exact-rotation splitting (stable)")

In [ ]:
# ---- (a) bias vs alpha, at the step size used for the mixing study -------------------------
H_OP = {"A": 0.005, "B": 0.0025}   # target B needs half the step: its kinks are far denser
TOL  = 0.04                         # ~3x the Monte-Carlo noise floor of the diagnostic below
ALPHAS = [1.0, 2.0, 4.0, 8.0]
bias = {}
for k, tg in TARGETS.items():
    bias[k] = {}
    cfgs = [("J0", 0.0, dict(J="none"))]
    cfgs += [("Ja", a, dict(J="const", alpha=a)) for a in ALPHAS]
    cfgs += [("Js", a, dict(J="state", alpha=a, s=S_STATE[k])) for a in ALPHAS]
    print(f"\n=== target {k}: relative stationarity drift over T=2 at h={H_OP[k]} ===")
    print(f"  {'':<12}" + "".join(f"{n:>11}" for n in ["x1^2","x2^2","x3^2","U"]) + f"{'max|.|':>10}")
    for tag, a, kw in cfgs:
        t0 = time.time(); m, se = stat_drift(tg, T=2.0, h=H_OP[k], M=120_000, **kw)
        bias[k][(tag, a)] = m
        flag = "   <-- flagged" if np.abs(m).max() > TOL else ""
        print(f"  {tag+(f' a={a:g}' if tag!='J0' else ''):<12}" + "".join(f"{v:>11.4f}" for v in m)
              + f"{np.abs(m).max():>10.4f}{flag}   (2se~{2*se.max():.4f}, {time.time()-t0:.0f}s)")

In [ ]:
# ---- bias is O(h) down to the Monte-Carlo floor of the diagnostic ----------------------------
HS_T   = {"A": [0.02, 0.01, 0.005], "B": [0.02, 0.01, 0.005, 0.0025]}
refine, refine_se = {}, {}
for k, tg in TARGETS.items():
    refine[k], refine_se[k] = {}, {}
    for tag, kw in [("J0", dict(J="none")),
                    ("Ja", dict(J="const", alpha=4.0)),
                    ("Js", dict(J="state", alpha=4.0, s=S_STATE[k]))]:
        out = [stat_drift(tg, T=2.0, h=hh, M=80_000, **kw) for hh in HS_T[k]]
        refine[k][tag]    = [np.abs(m).max() for m, _ in out]
        refine_se[k][tag] = [2*se.max()      for _, se in out]
        print(f"target {k}  {tag:<3}  max|drift| over h={HS_T[k]}:  "
              + "  ".join(f"{a:.4f}+-{b:.4f}" for a, b in zip(refine[k][tag], refine_se[k][tag])))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
for ax, k in zip(axes, TARGETS):
    hs = np.array(HS_T[k])
    for tag in ["J0", "Ja", "Js"]:
        ax.errorbar(hs, refine[k][tag], yerr=refine_se[k][tag], fmt="o-", color=COL[tag], lw=1.2, ms=4,
                    capsize=2, label={"J0": r"$J=0$", "Ja": r"$J_a,\ \alpha=4$", "Js": r"$J_s,\ \alpha=4$"}[tag])
    ax.axhspan(0, np.mean(refine_se[k]["J0"]), color="0.85", zorder=0)
    ax.plot(hs, hs/hs[0]*refine[k]["Js"][0], "k:", lw=.9, label=r"$O(h)$")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("step size $h$"); ax.set_ylabel(r"max$_f$ |relative drift|")
    ax.set_title(f"Target {k}: discretisation bias vs $h$"); ax.legend(fontsize=7)
fig.suptitle("shaded band = Monte-Carlo noise floor of the diagnostic (2 s.e.)", y=1.03, fontsize=8)
fig.tight_layout(); plt.show()

---
# 7. Experiment 2 — mixing

Now the actual comparison. Within each target all runs use the **same $h$, the same number of steps and
the same number of $\nabla U_0$ evaluations**, so integrated autocorrelation time in steps is directly
proportional to cost per effective sample. We report

$$\tau_f\ \text{(steps and simulated time)},\qquad
\text{ESS},\qquad
\text{speed-up}=\frac{\tau_f(J=0)}{\tau_f(J)},$$

for $f\in\{x_1,x_2,x_3,U\}$, alongside the measured bias from §6 so that the accuracy cost of each
$\alpha$ is visible in the same table. Target B runs at $h=0.0025$, half of target A's step: its
non-differentiable set is the union of all three coordinate hyperplanes rather than a single point,
and at a common $\delta$ that costs a factor of two in step size. Integrated autocorrelation times are
also reported in units of *simulated time* ($\tau h$), which is comparable across the two targets.

In [ ]:
RUN = {"A": dict(h=0.005, n_steps=600_000, burn=100_000, thin=10, n_chains=48),
       "B": dict(h=0.0025, n_steps=800_000, burn=160_000, thin=20, n_chains=48)}
names = ["x1", "x2", "x3", "U"]

def run_config(tg, k, tag, alpha):
    kw = {"J0": dict(J="none"),
          "Ja": dict(J="const", alpha=alpha),
          "Js": dict(J="state", alpha=alpha, s=S_STATE[k])}[tag]
    p  = RUN[k]
    t0 = time.time()
    tr, _ = nald(tg, seed=2024, **p, **kw)
    S  = summarize(tr, p["thin"], p["h"], names)
    S["acf"] = [ac[:6000] for ac in S["acf"]]
    n, K, m = tr.shape
    true = np.r_[0.0, 0.0, 0.0, tg.EU_exact()]
    cm   = np.cumsum(tr, axis=0) / np.arange(1, n+1)[:, None, None]      # running means, per chain
    S["mse"] = ((cm - true)**2).mean(axis=1)                             # (n, m) across-chain MSE
    S["trace_sub"] = tr[::5, :3, :].copy()                                      # for trace plots
    idx = np.random.default_rng(9).integers(0, n*K, size=200_000)
    S["flat_sub"] = tr.reshape(-1, m)[idx]
    S["tag"], S["alpha"], S["wall"] = tag, alpha, time.time()-t0
    del tr, cm
    return S

CFGS = [("J0", 0.0)] + [("Ja", a) for a in ALPHAS] + [("Js", a) for a in ALPHAS]
sweep = {}
for k, tg in TARGETS.items():
    sweep[k] = {}
    print(f"--- target {k}  (h={RUN[k]['h']}, {RUN[k]['n_steps']:,} steps, "
          f"T={RUN[k]['n_steps']*RUN[k]['h']:.0f}, {RUN[k]['n_chains']} chains) ---")
    for tag, a in CFGS:
        S = run_config(tg, k, tag, a); sweep[k][(tag, a)] = S
        print(f"  {tag}{'' if tag=='J0' else f' a={a:g}':<6}  tau(steps) = "
              + " ".join(f"{t:8.0f}" for t in S["tau_steps"]) + f"   [{S['wall']:5.1f}s]")

In [ ]:
# ---------------- master table ----------------
for k, tg in TARGETS.items():
    base = sweep[k][("J0", 0.0)]["tau_steps"]
    print(f"\n=== target {k}: {'elliptical Laplace' if k=='A' else 'l1 Laplace'} "
          f"(h={RUN[k]['h']}, T={RUN[k]['n_steps']*RUN[k]['h']:.0f} per chain) ===")
    print(f"  {'config':<12}" + "".join(f"{'tau_'+n:>9}" for n in names)
          + "  | " + "".join(f"{'sp_'+n:>8}" for n in names) + "  |  max|bias|")
    for tag, a in CFGS:
        S = sweep[k][(tag, a)]; sp = base/S["tau_steps"]
        b  = np.abs(bias[k][(tag, a)]).max()
        flag = " (*)" if b > TOL else ""
        print(f"  {tag+('' if tag=='J0' else f' a={a:g}'):<12}"
              + "".join(f"{t:>9.0f}" for t in S["tau_steps"])
              + "  | " + "".join(f"{v:>8.2f}" for v in sp)
              + f"  |   {b:>7.4f}{flag}")
    print("  tau in integrator steps (identical cost per step across configs); sp = speed-up over J=0.")
    print(f"  (*) marks configs whose measured stationarity drift exceeds TOL = {TOL:.0%} -- not trustworthy.")

In [ ]:
# ---------------- speed-up vs alpha, with the accuracy cost shown alongside ----------------
fig, axes = plt.subplots(2, 3, figsize=(12, 6.0))
for r, (k, tg) in enumerate(TARGETS.items()):
    base = sweep[k][("J0", 0.0)]["tau_steps"]
    for j, nm in enumerate(["x1", "x2", "U"]):
        jj = names.index(nm); ax = axes[r, j]
        for tag, lab in [("Ja", r"$J_a$ (constant)"), ("Js", r"$J_s$ (state dep.)")]:
            sp = [base[jj]/sweep[k][(tag, a)]["tau_steps"][jj] for a in ALPHAS]
            ok = [np.abs(bias[k][(tag, a)]).max() <= TOL for a in ALPHAS]
            ax.plot(ALPHAS, sp, "-", color=COL[tag], lw=1.3, label=lab)
            ax.plot(np.array(ALPHAS)[ok],  np.array(sp)[ok],  "o", color=COL[tag], ms=5)
            ax.plot(np.array(ALPHAS)[~np.array(ok)], np.array(sp)[~np.array(ok)], "x", color=COL[tag], ms=6)
        ax.axhline(1.0, color="0.5", lw=.8, ls=":")
        ax.set_xscale("log", base=2); ax.set_xticks(ALPHAS); ax.set_xticklabels([f"{a:g}" for a in ALPHAS])
        ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"speed-up  $\tau(J{=}0)/\tau(J)$")
        ax.set_title(f"Target {k} — $f={'U(x)' if nm=='U' else nm}$")
        if r == 0 and j == 0: ax.legend(fontsize=7)
fig.suptitle("Mixing speed-up vs perturbation strength   (o = bias within tolerance,  x = beyond it, untrustworthy)", y=1.01)
fig.tight_layout(); plt.show()

In [ ]:
# ---------------- pick the best trustworthy alpha per (target, J) and compare in detail ------
BEST = {}
for k in TARGETS:
    for tag in ["Ja", "Js"]:
        ok = [a for a in ALPHAS if np.abs(bias[k][(tag, a)]).max() <= TOL]
        # among admissible alphas, the one minimising the mean normalised tau over the 4 test functions
        base = sweep[k][("J0", 0.0)]["tau_steps"]
        BEST[(k, tag)] = min(ok, key=lambda a: np.mean(sweep[k][(tag, a)]["tau_steps"]/base)) if ok else ALPHAS[0]   # fall back to the weakest perturbation
    BEST[(k, "J0")] = 0.0
print("selected operating points (largest gain subject to the stationarity-drift tolerance):")
for k in TARGETS:
    print(f"  target {k}:  J_a alpha = {BEST[(k,'Ja')]},   J_s alpha = {BEST[(k,'Js')]}")

In [ ]:
# ---------------- autocorrelation functions ----------------
for k, tg in TARGETS.items():
    p = RUN[k]
    fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
    for j, nm in enumerate(names):
        for tag in ["J0", "Ja", "Js"]:
            a  = BEST[(k, tag)]; S = sweep[k][(tag, a)]
            ac = S["acf"][j]; lag_t = np.arange(len(ac))*p["thin"]*p["h"]
            lab = r"$J=0$" if tag == "J0" else (r"$J_a,\ \alpha=%g$" % a if tag == "Ja" else r"$J_s,\ \alpha=%g$" % a)
            axes[j].plot(lag_t, ac, color=COL[tag], lw=1.1, label=lab)
        axes[j].axhline(0, color="0.5", lw=.6)
        axes[j].set_xlim(0, 6*sweep[k][("J0",0.0)]["tau_time"][j])
        axes[j].set_xlabel("lag (simulated time)")
        axes[j].set_title(("$x_%d$" % (j+1)) if j < 3 else "$U(x)$")
    axes[0].set_ylabel("autocorrelation"); axes[0].legend(fontsize=7)
    fig.suptitle(f"Target {k} — autocorrelation at the selected operating points", y=1.05)
    fig.tight_layout(); plt.show()

In [ ]:
# ---------------- traces of the slowest coordinate ----------------
for k, tg in TARGETS.items():
    p = RUN[k]
    fig, axes = plt.subplots(3, 1, figsize=(11, 4.6), sharex=True, sharey=True)
    for ax, tag in zip(axes, ["J0", "Ja", "Js"]):
        a = BEST[(k, tag)]; S = sweep[k][(tag, a)]
        t = np.arange(S["trace_sub"].shape[0])*5*p["thin"]*p["h"]
        ax.plot(t, S["trace_sub"][:, 0, 0], lw=.35, color=COL[tag])
        lab = r"$J=0$" if tag == "J0" else (r"$J_a,\ \alpha=%g$" % a if tag == "Ja" else r"$J_s,\ \alpha=%g$" % a)
        ax.set_ylabel(lab, fontsize=8)
        ax.text(.995, .88, r"$\tau_{x_1}=%.0f$ steps" % S["tau_steps"][0], transform=ax.transAxes,
                ha="right", fontsize=7.5)
    axes[-1].set_xlabel("simulated time $t$")
    fig.suptitle(f"Target {k} — trace of the slow coordinate $x_1$ (single chain)", y=.99)
    fig.tight_layout(); plt.show()

In [ ]:
# ---------------- MSE of the ergodic average ----------------
fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.0))
for r, (k, tg) in enumerate(TARGETS.items()):
    p = RUN[k]
    for j, nm in enumerate(["x1", "U"]):
        jj = names.index(nm); ax = axes[r, j]
        for tag in ["J0", "Ja", "Js"]:
            a = BEST[(k, tag)]; S = sweep[k][(tag, a)]
            t = (np.arange(1, S["mse"].shape[0]+1))*p["thin"]*p["h"]
            lab = r"$J=0$" if tag == "J0" else (r"$J_a,\ \alpha=%g$" % a if tag == "Ja" else r"$J_s,\ \alpha=%g$" % a)
            ax.loglog(t, S["mse"][:, jj], color=COL[tag], lw=1.2, label=lab)
        ax.set_xlabel("simulated time of the ergodic average"); ax.set_ylabel("MSE across chains")
        ax.set_title(f"Target {k} — $\\widehat{{\\mathbb{{E}}}}[{'U(x)' if nm=='U' else nm}]$")
        if r == 0 and j == 0: ax.legend(fontsize=7)
fig.suptitle("Mean-squared error of the ergodic average (48 independent chains)", y=1.01)
fig.tight_layout(); plt.show()

---
# 8. Experiment 3 — why $J_s$ behaves the way it does

Recall from §3 that with the canonical $\psi$,

$$\alpha c(x)=\bigl(\alpha s\,a(x)\nabla U_0(x)\bigr)\times x .$$

So the state-dependent perturbation is a rotation about the axis $\nabla U_0(x)$, and it
**vanishes identically wherever $x$ is parallel to $\nabla U_0(x)$**.

For the elliptical target $\nabla U_0(x)\propto\Sigma^{-1}x$, so $x\times\nabla U_0=0$ exactly on the
**principal axes of $\Sigma$** — which is precisely the slow manifold the sampler needs to travel along.
$J_s$ therefore switches itself off on the very directions where acceleration is wanted, and instead
stirs the fast, orthogonal directions.

For the $\ell^1$ target $\nabla U_0\approx(\mathrm{sign}(x_i)/b_i)_i$ is nearly constant inside each
orthant, so the degeneracy is a single ray per orthant rather than the coordinate axes, but the same
mechanism holds.

The constant $J_a v=u\times v$ has no such degeneracy along the principal axes (as long as $u$ is not an
eigenvector of $\Sigma$), which is why it is the more effective accelerator of the slow coordinate here.

In [ ]:
# magnitude of the antisymmetric drift relative to the reversible drift, on the (x1,x2) slice
fig, axes = plt.subplots(2, 3, figsize=(12, 6.6))
for r, (k, tg) in enumerate(TARGETS.items()):
    lim = 3*np.sqrt(np.diag(tg.cov_exact())[0])
    g1 = np.linspace(-lim, lim, 261)
    X1, X2 = np.meshgrid(g1, g1)
    P = np.stack([X1.ravel(), X2.ravel(), np.zeros(X1.size)], -1)
    a = np.exp(tg.log_a(P))[:, None]; g = tg.gradU0(P)
    rev = a*g
    pa  = a*(g @ J_A.T)                       # J_a grad U0
    ps  = S_STATE[k]*a*np.cross(P, g)         # J_s grad U0
    nr  = np.linalg.norm(rev, axis=1)
    for j, (fld, ttl) in enumerate([(np.linalg.norm(pa,axis=1)/nr, r"$\|J_a\nabla U_0\|\,/\,\|\nabla U_0\|$"),
                                    (np.linalg.norm(ps,axis=1)/nr, r"$\|J_s(x)\nabla U_0\|\,/\,\|\nabla U_0\|$")]):
        ax = axes[r, j]
        im = ax.pcolormesh(X1, X2, fld.reshape(X1.shape), cmap="magma", shading="auto",
                           vmin=0, vmax=np.quantile(fld, .995))
        ax.contour(X1, X2, tg.U(np.stack([X1,X2,np.zeros_like(X1)],-1)), levels=6, colors="w", linewidths=.5, alpha=.6)
        plt.colorbar(im, ax=ax, fraction=.046)
        ax.set_title(f"Target {k}: {ttl}", fontsize=9); ax.set_aspect("equal")
        ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    # drift field of the best J_a configuration
    ax = axes[r, 2]
    q  = np.linspace(-lim, lim, 21); Q1, Q2 = np.meshgrid(q, q)
    Pq = np.stack([Q1.ravel(), Q2.ravel(), np.zeros(Q1.size)], -1)
    aq = np.exp(tg.log_a(Pq))[:, None]; gq = tg.gradU0(Pq)
    bq = -aq*gq - BEST[(k,"Ja")]*aq*(gq @ J_A.T)
    ax.contour(X1, X2, tg.U(np.stack([X1,X2,np.zeros_like(X1)],-1)), levels=8, colors="0.75", linewidths=.6)
    ax.quiver(Q1, Q2, bq[:,0].reshape(Q1.shape), bq[:,1].reshape(Q1.shape), color="#BF616A",
              width=.004, scale=None)
    ax.set_title(r"Target %s: drift $b_\alpha$, $J_a$, $\alpha=%g$ ($x_3=0$ slice)" % (k, BEST[(k,'Ja')]), fontsize=9)
    ax.set_aspect("equal"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
fig.tight_layout(); plt.show()
print("The J_s panel is dark (perturbation ~ 0) exactly along the principal axes -- the slow directions.")

In [ ]:
# quantitative version: perturbation strength restricted to the slow direction e1
print("||J grad U0|| / ||grad U0||  along the slow axis  x = (t, 0, 0):\n")
for k, tg in TARGETS.items():
    t = np.array([0.5, 1.0, 2.0, 4.0, 8.0])
    P = np.stack([t, np.zeros_like(t), np.zeros_like(t)], -1)
    g = tg.gradU0(P); ng = np.linalg.norm(g, axis=1)
    ra = np.linalg.norm(g @ J_A.T, axis=1)/ng
    rs = np.linalg.norm(S_STATE[k]*np.cross(P, g), axis=1)/ng
    print(f"  target {k}:   t        = " + "".join(f"{v:>9.2f}" for v in t))
    print(f"              J_a      = " + "".join(f"{v:>9.4f}" for v in ra))
    print(f"              J_s(x)   = " + "".join(f"{v:>9.4f}" for v in rs))
    print()

---
# 9. Summary

In [ ]:
print("="*104)
print("NALD on non-differentiable targets -- summary".center(104))
print("="*104)
for k, tg in TARGETS.items():
    p = RUN[k]; base = sweep[k][("J0", 0.0)]
    print(f"\nTarget {k}: {'elliptical Laplace  U(x)=sqrt(x^T S^-1 x)' if k=='A' else 'l1 Laplace  U(x)=sum |x_i|/b_i'}"
          f"   |  anchor delta={tg.delta},  h={p['h']},  T={p['n_steps']*p['h']:.0f},  {p['n_chains']} chains")
    print(f"  {'':<22}" + "".join(f"{'tau_'+n+' (time)':>16}" for n in names) + f"{'max|bias|':>12}")
    for tag in ["J0", "Ja", "Js"]:
        a = BEST[(k, tag)]; S = sweep[k][(tag, a)]
        lab = "J = 0 (reversible)" if tag == "J0" else (f"J_a  alpha={a:g}" if tag == "Ja" else f"J_s  alpha={a:g}")
        sp  = base["tau_steps"]/S["tau_steps"]
        print(f"  {lab:<22}" + "".join(f"{S['tau_time'][j]:>9.2f} ({sp[j]:>4.2f}x)" for j in range(4))
              + f"{np.abs(bias[k][(tag,a)]).max():>12.4f}")
    # accuracy line
    ex = np.r_[0,0,0, tg.EU_exact()]; exv = np.r_[np.diag(tg.cov_exact()), np.nan]
    for tag in ["J0", "Ja", "Js"]:
        f = sweep[k][(tag, BEST[(k,tag)])]["flat_sub"]
        print(f"     {tag:<4} Var(x) = {np.round(f[:,:3].var(0),3)}  (exact {np.round(exv[:3],3)}),"
              f"   E[U] = {f[:,3].mean():.4f}  (exact {tg.EU_exact():.4f})")
print("\n" + "="*104)


### What the experiments show

1. **Anchoring works.** Both targets are non-differentiable ($U$ has a conical cusp at the origin in
   Target A and kinks on every coordinate hyperplane in Target B), yet NALD never evaluates $\nabla U$.
   Replacing it by a $C^\infty$ anchor $U_0$ together with the multiplier $a=e^{U-U_0}$ reproduces the
   exact target: marginals, Q-Q plots, $\mathrm{Var}(x_i)$ and $\mathbb{E}[U]$ all match i.i.d. draws to
   Monte-Carlo error, and the paired stationarity test confirms invariance to within the $O(h)$
   discretisation error. **The invariant law does not depend on the smoothing parameter $\delta$**
   — with common random numbers the measured drift is unchanged to four decimals across a 50-fold range
   of $\delta$. This is what distinguishes anchoring from simply sampling a smoothed surrogate
   $e^{-U_0}$, which would be biased by $O(\delta)$.

2. **$\delta$ is a free knob, but not a free lunch.** It cannot bias the answer, so it can be tuned
   purely for efficiency — and there is a real optimum: small $\delta$ keeps $a\approx1$ and mixes
   fast but stiffens $\nabla U_0$; large $\delta$ conditions the drift but collapses $a$ and slows the
   diffusion (a $3.7\times$ increase in $\tau_{x_1}$ from $\delta=0.1$ to $\delta=1$ on Target B).

3. **Both non-reversible perturbations preserve $\pi$ and accelerate mixing** in the regime where the
   discretisation is accurate. $J_a$ is the better accelerator of the *slow* coordinate on both
   targets; $J_s$ gives much larger gains on the fast coordinates but little on the slow one, and
   essentially none on $U(x)$.

4. **The mechanism behind (3).** With the canonical $\psi$, $\alpha c=(\alpha s\,a\nabla U_0)\times x$
   vanishes wherever $x\parallel\nabla U_0$ — on the principal axes of $\Sigma$ for Target A, i.e.
   exactly the directions that limit mixing (the table in §8 shows $\|J_s\nabla U_0\|/\|\nabla U_0\|=0$
   identically along $e_1$, against $0.8165$ for $J_a$). A constant $J_a$ has no such degeneracy. If a
   state-dependent $J$ is wanted, it should be designed so that $J(x)\nabla U_0(x)$ does *not*
   degenerate on the slow manifold.

5. **Non-reversibility is not free at the discrete level.** The stationarity drift grows with $\alpha$
   at fixed $h$ and is $O(h)$ at fixed $\alpha$, so large $\alpha$ must be paid for with proportionally
   smaller $h$; past a target-dependent threshold the apparent "speed-up" is just bias. Every table
   above therefore reports the measured bias next to the speed-up, and flags the configurations where
   the speed-up should not be believed.

6. **State-dependent $J$ needs a structure-preserving integrator.** The exact $J_s$-flow is a rotation
   and conserves $\|x\|$; plain Euler–Maruyama inflates $\|x\|$ by $\sqrt{1+(h\|\omega\|)^2}$ per step
   against a *bounded* restoring drift, and the chain escapes to infinity for moderate $\alpha$. A
   Lie–Trotter split with the rotation integrated exactly (Rodrigues) costs the same one gradient per
   step and removes the instability entirely.

### Reproducing / extending

* `nald(...)` takes any target object exposing `U`, `U0`, `gradU0`, `log_a`, `sample`; adding a new
  target is a dozen lines.
* The general form $c=e^{U}J\nabla\psi$ for a non-canonical $\psi$ slots into the same integrator — only
  the line computing `w` / the `const` drift changes. The canonical $\psi=e^{-U_0}$ is used here because
  it makes $c=-aJ\nabla U_0$ overflow-free.
* Natural next steps: a joint $(\alpha,h)$ sweep at matched accuracy rather than matched step count;
  an anisotropy-adapted constant $J$; a state-dependent $J$ built to be non-degenerate along the slow
  manifold; and a $\delta$-annealing schedule.
